# 01 · Prepare the Hinglish corpus

Runs the data pipeline (download → clean → MinHash dedup → split) on Colab or Kaggle and
stores the result somewhere that survives a disconnect (Google Drive on Colab,
`/kaggle/working` on Kaggle — save it as a Kaggle Dataset afterwards).

Sources and licenses are in `configs/data.yaml`. **HingCorpus is CC BY-NC-SA 4.0**: fine for
this non-commercial learning project; anything derived from it (e.g. model weights) must
carry the same license.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

# Work on the runtime's FAST local disk; Google Drive is very slow for writing millions of
# small lines. Only the big HingCorpus download (on Drive) and the final results are kept.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    KEEP = "/content/drive/MyDrive/hinglish-nanolm/data"   # survives disconnects
    DATA = "/content/data"                                 # fast, wiped on disconnect
elif IN_KAGGLE:
    KEEP = DATA = "/kaggle/working/hinglish-nanolm/data"
else:
    KEEP = DATA = os.path.abspath("../data")
os.makedirs(f"{KEEP}/downloads", exist_ok=True)
os.makedirs(DATA, exist_ok=True)
if KEEP != DATA and not os.path.exists(f"{DATA}/downloads"):
    os.symlink(f"{KEEP}/downloads", f"{DATA}/downloads")   # read downloads straight from Drive
print("working dir:", DATA, "| results saved to:", KEEP)

In [ ]:
# Get the code. If the repo is private, use a GitHub token:
#   https://<token>@github.com/naqi7272/Hinglish-nanoLM.git
REPO = "https://github.com/naqi7272/Hinglish-nanoLM.git"
BRANCH = "claude/hinglish-nanolm-setup-finaye"
if IN_COLAB or IN_KAGGLE:
    %cd /content
    !git clone -q -b {BRANCH} {REPO} code || (cd code && git pull -q)
    %cd /content/code
    !pip install -q -e ".[data]"

## 1. Download HingCorpus (Google Drive link from the L3Cube README)
It arrives as one 2.4 GB `L3Cube-HingCorpus_roman.tar.gz`. We leave it compressed.


In [ ]:
HC = f"{KEEP}/downloads/hingcorpus"
os.makedirs(HC, exist_ok=True)
if not os.listdir(HC):
    !gdown -q --fuzzy "https://drive.google.com/file/d/1s_6eHO9zDhxQ-xVN1TyNguszV1meZkl9/view" -O {HC}/
!ls -lh {HC}

In [ ]:
# Look inside the archive WITHOUT unpacking it (unpacking would need ~5 GB more Drive space).
# The pipeline streams documents straight out of the .tar.gz.
import glob
archive = glob.glob(f"{HC}/*.tar.gz")[0]
print(archive)
!tar -tzvf "$archive" | head -n 5
!tar -xzOf "$archive" | head -n 5


## 2. Peek at every source
Confirms the `text_field` / `data_dir` values in `configs/data.yaml` (the ones marked *verify*).
If a field is wrong, the error message lists the real column names.

In [ ]:
!python scripts/prepare_data.py --data-dir {DATA} --stages peek

## 3. Run the pipeline (~20-40 min, CPU runtime is fine — a GPU doesn't help here)
Every stage skips work that's already finished, so after a disconnect just re-run the
setup cells and this cell again.

In [ ]:
!python scripts/prepare_data.py --data-dir {DATA} --stages download,clean,dedup,split,stats

In [ ]:
# Save the results to Drive (the local disk is wiped when the runtime disconnects).
import json, shutil
shutil.copytree(f"{DATA}/splits", f"{KEEP}/splits", dirs_exist_ok=True)
shutil.copy(f"{DATA}/stats.json", f"{KEEP}/stats.json")
print(json.dumps(json.load(open(f"{KEEP}/stats.json")), indent=2, ensure_ascii=False))
!ls -lh {KEEP}/splits